# Pokémon TCG — Supervised Value Network

Trains an encoder-only Transformer to predict game outcome (win=+1, loss=−1) from board state.  
Labels are either raw terminal results or TD(λ)-smoothed estimates using the current model.  

**Folder layout expected at runtime:**
```
SupervisedValueNetwork/
  value_network_training.ipynb   ← this file
  Models/                        ← saved checkpoints
  pngs/                          ← loss + parity plots
  data_split.json                ← reproducible train/val/test split
../Training-Data/                ← raw game JSON files
../features.py                   ← feature extraction
```

## 1 — Configuration
Change `MODEL_NAME` for each experiment. Everything else flows from it.

In [ ]:
# ── Experiment identity ───────────────────────────────────────────────────────
MODEL_NAME    = "FirstTry100Epochs"   # used for checkpoint and plot filenames

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_NAME  = "6-16"           # choose from: "6-16 only", "6-16_6-17"
DATA_DIR      = f"../Training-Data/{DATASET_NAME}"
MODEL_DIR     = "Models"
PNG_DIR       = "pngs"
SPLIT_FILE    = "data_split.json"  # saved once; reloaded on subsequent runs

# ── Data split ───────────────────────────────────────────────────────────────
VAL_FRACTION  = 0.10
TEST_FRACTION = 0.10
SEED          = 42

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS        = 100  # early stopping may stop earlier
BATCH_SIZE    = 128
LR            = 3e-4
USE_TD_LAMBDA = True   # False = raw terminal result for every step
LAMBDA        = 0.9

# ── Model ────────────────────────────────────────────────────────────────────
D_MODEL           = 128
NUM_HEADS         = 2
D_FEEDFORWARD     = 256
NUM_ENCODER_LAYERS = 2
ENCODER_SIZE      = 22000
DROPOUT           = 0.2

# ── Early stopping ───────────────────────────────────────────────────────────
PATIENCE          = 5   # stop if val_loss doesn't improve for N epochs

## 2 — Imports & Directory Setup

In [ ]:
import glob
import json
import os
import random
import sys

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# features.py lives one level up
sys.path.insert(0, os.path.abspath(".."))
from features import SparseVector, get_encoder_input, CARD_COUNT, NUM_WORDS_ENCODER

# ── Create output directories ─────────────────────────────────────────────────
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(os.path.join(PNG_DIR, MODEL_NAME), exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Model  : {MODEL_NAME}")
print(f"Plots  → pngs/{MODEL_NAME}/")
print(f"Models → {MODEL_DIR}/")

## 3 — Model Definition
Encoder-only Transformer. Architecture matches the encoder path of `MyModel` in the MCTS sample notebook so weights are transferable.

In [ ]:
class ValueModel(nn.Module):
    """
    Transformer encoder → scalar board value in [−1, +1].

    Input  : 24 sparse board-state words (via EmbeddingBag)
    Output : single float — positive means 'you are winning'
    """
    def __init__(self,
                 d_model=D_MODEL,
                 num_heads=NUM_HEADS,
                 d_feedforward=D_FEEDFORWARD,
                 num_layers=NUM_ENCODER_LAYERS,
                 encoder_size=ENCODER_SIZE,
                 dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.encoder_bag = nn.EmbeddingBag(encoder_size, d_model, mode="sum")
        enc_layer = nn.TransformerEncoderLayer(
            d_model, num_heads, d_feedforward, dropout=dropout, batch_first=False
        )
        self.encoder = nn.TransformerEncoder(
            enc_layer, num_layers, enable_nested_tensor=False
        )
        self.fc = nn.Linear(d_model, 1)

    def forward(self, index, value, offset):
        """
        index  : int32  flat nonzero indices  (sum of all words in batch)
        value  : float32 weights for each index
        offset : int32  start of each word in `index`  (24 * batch_size entries)
        Returns: (batch_size, 1) tensor in [−1, +1]
        """
        x = self.encoder_bag(index, offset, value)          # (24*B, d_model)
        x = x.reshape(-1, NUM_WORDS_ENCODER, self.d_model).transpose(0, 1)  # (24, B, d_model)
        x = self.encoder(x)                                  # (24, B, d_model)
        x = torch.tanh(self.fc(x).mean(0))                  # (B, 1)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = ValueModel().to(device)
print(f"Parameters: {count_parameters(model):,}")

## 4 — Data Structures & Loading

In [ ]:
class GameSample:
    __slots__ = ("sv", "label")
    def __init__(self, sv: SparseVector, label: float):
        self.sv    = sv
        self.label = label


class Trajectory:
    """One player's ordered observations from a single game."""
    __slots__ = ("svs", "terminal")
    def __init__(self, svs: list, terminal: float):
        self.svs      = svs       # list[SparseVector], chronological
        self.terminal = terminal  # +1 win / −1 loss / 0 draw


def load_game_trajectories(path: str) -> list:
    """Load per-player ordered observations from one game file."""
    try:
        with open(path) as f:
            d = json.load(f)
    except Exception:
        return []

    rewards = d.get("rewards", [0, 0])
    steps   = d.get("steps", [])
    if len(steps) < 2:
        return []

    decks = [[], []]
    for pi in range(2):
        action   = steps[1][pi].get("action") or []
        decks[pi] = [c for c in action if isinstance(c, int)]

    per_player = [[], []]
    for step in steps:
        for pi in range(2):
            obs     = step[pi].get("observation", {})
            current = obs.get("current")
            if current is None:
                continue
            your_index = current.get("yourIndex")
            if your_index is None:
                continue
            deck = decks[your_index]
            if not deck:
                continue
            try:
                sv = get_encoder_input(obs, deck)
                per_player[your_index].append(sv)
            except Exception:
                continue

    out = []
    for pi in range(2):
        if per_player[pi] and rewards[pi] is not None:
            out.append(Trajectory(per_player[pi], float(rewards[pi])))
    return out


def load_flat_game(path: str) -> list:
    """Simple flat loader — each step gets the terminal result as label."""
    samples = []
    for traj in load_game_trajectories(path):
        for sv in traj.svs:
            samples.append(GameSample(sv, traj.terminal))
    return samples


print("Data structures defined.")

## 5 — Train / Val / Test Split
Split is saved to `data_split.json` on first run and reloaded on subsequent runs, guaranteeing the test set is never seen during training.

In [ ]:
all_files = sorted(glob.glob(os.path.join(os.path.abspath(DATA_DIR), "*.json")))

if os.path.exists(SPLIT_FILE):
    with open(SPLIT_FILE) as f:
        split = json.load(f)
    train_files = split["train"]
    val_files   = split["val"]
    test_files  = split["test"]
    # Validate that the stored paths actually exist; regenerate if they don't
    if not os.path.exists(train_files[0]):
        print(f"WARNING: Split paths are stale (e.g. {train_files[0]!r}). Regenerating…")
        os.remove(SPLIT_FILE)
        train_files = val_files = test_files = None
    else:
        print(f"Loaded existing split from {SPLIT_FILE}")

if not os.path.exists(SPLIT_FILE):
    random.shuffle(all_files)
    n        = len(all_files)
    n_test   = max(1, int(n * TEST_FRACTION))
    n_val    = max(1, int(n * VAL_FRACTION))
    test_files  = all_files[:n_test]
    val_files   = all_files[n_test : n_test + n_val]
    train_files = all_files[n_test + n_val :]
    with open(SPLIT_FILE, "w") as f:
        json.dump({"train": train_files, "val": val_files, "test": test_files}, f, indent=2)
    print(f"Created new split → saved to {SPLIT_FILE}")

print(f"Train : {len(train_files):4d} games")
print(f"Val   : {len(val_files):4d} games")
print(f"Test  : {len(test_files):4d} games")

## 6 — Load Trajectories

In [ ]:
def load_trajectories(files: list, label: str) -> list:
    trajs = []
    for path in tqdm(files, desc=f"Loading {label}", unit="game"):
        trajs.extend(load_game_trajectories(path))
    total_steps = sum(len(t.svs) for t in trajs)
    print(f"{label:6s}: {len(trajs):4d} trajectories, {total_steps:7,d} observations")
    return trajs

train_trajs = load_trajectories(train_files, "Train")
val_trajs   = load_trajectories(val_files,   "Val")
test_trajs  = load_trajectories(test_files,  "Test")

## 7 — Dataset, Collation & TD(λ)

In [ ]:
# ── Batch packing ────────────────────────────────────────────────────────────

def collate_fn(batch: list):
    flat_idx, flat_val, flat_off, labels = [], [], [], []
    for s in batch:
        base = len(flat_idx)
        flat_idx.extend(s.sv.index)
        flat_val.extend(s.sv.value)
        for o in s.sv.offset:
            flat_off.append(o + base)
        labels.append(s.label)
    return (
        torch.tensor(flat_idx, dtype=torch.int32),
        torch.tensor(flat_val, dtype=torch.float32),
        torch.tensor(flat_off, dtype=torch.int32),
        torch.tensor(labels,   dtype=torch.float32).unsqueeze(1),
    )


# ── TD(λ) label computation ───────────────────────────────────────────────────

def _pack_svs(svs: list):
    flat_idx, flat_val, flat_off = [], [], []
    for sv in svs:
        base = len(flat_idx)
        flat_idx.extend(sv.index)
        flat_val.extend(sv.value)
        for o in sv.offset:
            flat_off.append(o + base)
    return (
        torch.tensor(flat_idx, dtype=torch.int32),
        torch.tensor(flat_val, dtype=torch.float32),
        torch.tensor(flat_off, dtype=torch.int32),
    )


@torch.no_grad()
def batch_predict(svs: list, model, device, chunk=512) -> list:
    """Forward pass on a flat list of SparseVectors; returns Python floats."""
    model.eval()
    out = []
    for i in range(0, len(svs), chunk):
        idx, val, off = _pack_svs(svs[i:i+chunk])
        preds = model(idx.to(device), val.to(device), off.to(device))
        out.extend(preds.squeeze(1).cpu().tolist())
    return out


def apply_td_lambda(trajectories: list, model, device, lambda_: float = 0.9) -> list:
    """
    Compute TD(λ) labels.

    Walking backward through each trajectory:
        label[t]  = 0.5 * (carry + model_pred[t])
        carry     = λ * carry + (1 − λ) * model_pred[t]
    carry is initialised to the terminal result.
    """
    all_svs   = [sv for t in trajectories for sv in t.svs]
    all_preds = batch_predict(all_svs, model, device)

    samples, cursor = [], 0
    for traj in trajectories:
        n     = len(traj.svs)
        preds = all_preds[cursor:cursor+n]
        cursor += n

        carry  = traj.terminal
        labels = [0.0] * n
        for t in reversed(range(n)):
            labels[t] = (carry + preds[t]) * 0.5
            carry     = lambda_ * carry + (1 - lambda_) * preds[t]

        for sv, lbl in zip(traj.svs, labels):
            samples.append(GameSample(sv, lbl))

    return samples


# ── Dataset wrapper ───────────────────────────────────────────────────────────

class ValueDataset(Dataset):
    """
    Wraps a list of trajectories.
    Call .relabel(model, device) at the start of each epoch when USE_TD_LAMBDA=True.
    Call .relabel_flat() for plain terminal-result labels.
    """
    def __init__(self, trajectories: list):
        self.trajectories = trajectories
        self.samples: list = []

    def relabel(self, model, device, lambda_: float = LAMBDA):
        self.samples = apply_td_lambda(self.trajectories, model, device, lambda_)
        random.shuffle(self.samples)

    def relabel_flat(self):
        self.samples = [
            GameSample(sv, traj.terminal)
            for traj in self.trajectories
            for sv in traj.svs
        ]
        random.shuffle(self.samples)

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


train_ds = ValueDataset(train_trajs)
val_ds   = ValueDataset(val_trajs)
test_ds  = ValueDataset(test_trajs)
print("Datasets ready.")

## 8 — Training Loop
Saves two checkpoints each epoch:
- `Models/{MODEL_NAME}_best.pth` — lowest validation loss so far
- `Models/{MODEL_NAME}_latest.pth` — end of every epoch

In [ ]:
def run_epoch(loader, model, optimizer, loss_fn, device, train: bool, desc: str = ""):
    model.train() if train else model.eval()
    total_loss, n_batches = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        bar = tqdm(loader, desc=desc, unit="batch", leave=False)
        for idx, val, off, lbl in bar:
            idx, val, off, lbl = idx.to(device), val.to(device), off.to(device), lbl.to(device)
            pred = model(idx, val, off)
            loss = loss_fn(pred, lbl)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            n_batches  += 1
            bar.set_postfix(loss=f"{total_loss/n_batches:.4f}")
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def directional_accuracy(loader, model, device):
    """Fraction of non-draw steps where sign(prediction) == sign(label)."""
    model.eval()
    correct, total = 0, 0
    for idx, val, off, lbl in loader:
        idx, val, off, lbl = idx.to(device), val.to(device), off.to(device), lbl.to(device)
        pred  = model(idx, val, off)
        mask  = lbl.sign() != 0
        correct += (pred.sign()[mask] == lbl.sign()[mask]).sum().item()
        total   += mask.sum().item()
    return correct / total if total else 0.0


def train_model(model, train_ds, val_ds, epochs, batch_size, lr, device,
                use_td=USE_TD_LAMBDA, lambda_=LAMBDA,
                model_name=MODEL_NAME, model_dir=MODEL_DIR,
                patience=PATIENCE):

    optimizer   = optim.AdamW(model.parameters(), lr=lr)
    loss_fn     = nn.HuberLoss(delta=0.2)
    best_path   = os.path.join(model_dir, f"{model_name}_best.pth")
    latest_path = os.path.join(model_dir, f"{model_name}_latest.pth")

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss    = float("inf")
    best_epoch       = 1
    patience_counter = 0

    epoch_bar = tqdm(range(1, epochs + 1), desc="Epochs", unit="epoch")
    for epoch in epoch_bar:
        # ── Relabel ────────────────────────────────────────────────────────────
        if use_td:
            train_ds.relabel(model, device, lambda_)
            val_ds.relabel(model, device, lambda_)
        else:
            if epoch == 1:
                train_ds.relabel_flat()
                val_ds.relabel_flat()

        train_loader = DataLoader(train_ds, batch_size=batch_size,
                                  shuffle=False, collate_fn=collate_fn)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                                  shuffle=False, collate_fn=collate_fn)

        # ── Train + eval ───────────────────────────────────────────────────────
        train_loss = run_epoch(train_loader, model, optimizer, loss_fn, device,
                               train=True,  desc=f"E{epoch:02d} train")
        val_loss   = run_epoch(val_loader,   model, optimizer, loss_fn, device,
                               train=False, desc=f"E{epoch:02d} val  ")
        val_acc    = directional_accuracy(val_loader, model, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # ── Checkpoints ────────────────────────────────────────────────────────
        torch.save(model.state_dict(), latest_path)
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_epoch       = epoch
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
            marker = " ◀ best"
        else:
            patience_counter += 1
            marker = f" (no improvement {patience_counter}/{patience})"

        epoch_bar.set_postfix(
            train=f"{train_loss:.4f}",
            val=f"{val_loss:.4f}",
            acc=f"{100*val_acc:.1f}%"
        )
        tqdm.write(
            f"Epoch {epoch:03d}/{epochs}  "
            f"train={train_loss:.4f}  val={val_loss:.4f}  "
            f"acc={100*val_acc:.1f}%{marker}"
        )

        if patience_counter >= patience:
            tqdm.write(f"\nEarly stopping: val_loss hasn't improved for {patience} epochs.")
            break

    print(f"\nBest val loss {best_val_loss:.4f} at epoch {best_epoch}")
    print(f"Best model   → {best_path}")
    print(f"Latest model → {latest_path}")
    return history


print("Training functions defined.")

## 9 — Run Training

In [ ]:
history = train_model(
    model      = model,
    train_ds   = train_ds,
    val_ds     = val_ds,
    epochs     = EPOCHS,
    batch_size = BATCH_SIZE,
    lr         = LR,
    device     = device,
)

## 10 — Loss Curves

In [ ]:
def plot_loss(history: dict, model_name: str, png_dir: str):
    epochs  = range(1, len(history["train_loss"]) + 1)
    best_ep = int(np.argmin(history["val_loss"])) + 1

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{model_name} — Training History", fontsize=13, fontweight="bold")

    # ── Loss ──────────────────────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(epochs, history["train_loss"], label="Train loss",  color="#3b82f6", linewidth=2)
    ax.plot(epochs, history["val_loss"],   label="Val loss",    color="#f59e0b", linewidth=2)
    ax.axvline(best_ep, color="#22c55e", linestyle="--", linewidth=1.2, label=f"Best epoch ({best_ep})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Huber Loss")
    ax.set_title("Loss vs Epoch")
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)

    # ── Directional accuracy ───────────────────────────────────────────────────
    ax = axes[1]
    ax.plot(epochs, [100 * a for a in history["val_acc"]],
            color="#ec4899", linewidth=2, label="Val accuracy")
    ax.axhline(50, color="#94a3b8", linestyle=":", linewidth=1, label="Chance (50%)")
    ax.axvline(best_ep, color="#22c55e", linestyle="--", linewidth=1.2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Directional Accuracy (%)")
    ax.set_title("Win Prediction Accuracy vs Epoch")
    ax.set_ylim(40, 100)
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(png_dir, model_name, "loss.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

plot_loss(history, MODEL_NAME, PNG_DIR)

## 11 — Parity / Calibration Plot
Bins model predictions by value and shows the actual win rate in each bin.  
A perfectly calibrated model would follow the diagonal (predictions of +0.6 → ~80% win rate).

In [ ]:
@torch.no_grad()
def collect_predictions(trajectories: list, model, device, chunk=512):
    """Return (predictions, terminal_results) arrays over all observations."""
    model.eval()
    all_svs      = [sv for t in trajectories for sv in t.svs]
    all_terminals = [t.terminal for t in trajectories for _ in t.svs]
    all_preds    = batch_predict(all_svs, model, device, chunk)
    return np.array(all_preds), np.array(all_terminals)


def plot_parity(trajectories: list, model, device, model_name: str, png_dir: str,
                n_bins: int = 20):
    preds, terminals = collect_predictions(trajectories, model, device)

    # Map terminals: +1 → win=1, −1 → win=0  (exclude draws)
    mask   = terminals != 0
    preds  = preds[mask]
    wins   = (terminals[mask] == 1).astype(float)

    bin_edges  = np.linspace(-1, 1, n_bins + 1)
    bin_centres, win_rates, counts = [], [], []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (preds >= lo) & (preds < hi)
        if in_bin.sum() < 5:
            continue
        bin_centres.append((lo + hi) / 2)
        win_rates.append(wins[in_bin].mean())
        counts.append(in_bin.sum())

    bin_centres = np.array(bin_centres)
    win_rates   = np.array(win_rates)
    counts      = np.array(counts, dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{model_name} — Calibration & Prediction Distribution",
                 fontsize=13, fontweight="bold")

    # ── Calibration curve ─────────────────────────────────────────────────────
    ax = axes[0]
    # ideal diagonal: map prediction [-1,1] → win rate [0,1]
    diag_x = np.linspace(-1, 1, 100)
    diag_y = (diag_x + 1) / 2
    ax.plot(diag_x, diag_y, color="#94a3b8", linestyle="--", linewidth=1.2, label="y=x")
    sc = ax.scatter(bin_centres, win_rates, c=counts, cmap="Blues",
                    s=80, edgecolors="#1e293b", linewidths=0.5, zorder=3, label="Binned win rate")
    ax.scatter(bin_centres, win_rates, color="#3b82f6", linewidth=1.5, alpha=0.7)
    plt.colorbar(sc, ax=ax, label="Observations per bin")
    ax.set_xlim(-1, 1)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Model prediction")
    ax.set_ylabel("Actual win rate")
    ax.set_title("Calibration: Prediction → Win Rate")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ── Prediction histogram split by outcome ─────────────────────────────────
    ax = axes[1]
    all_preds_full, all_terminals_full = collect_predictions(trajectories, model, device)
    w_preds = all_preds_full[all_terminals_full ==  1]
    l_preds = all_preds_full[all_terminals_full == -1]
    bins = np.linspace(-1, 1, 40)
    ax.hist(w_preds, bins=bins, alpha=0.6, color="#22c55e", label="Won games",  density=True)
    ax.hist(l_preds, bins=bins, alpha=0.6, color="#ef4444", label="Lost games", density=True)
    ax.axvline(0, color="#1e293b", linewidth=1)
    ax.set_xlabel("Model prediction")
    ax.set_ylabel("Density")
    ax.set_title("Prediction Distribution by Outcome")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(png_dir, model_name, "parity.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved → {path}")

# Load best checkpoint before plotting
best_path = os.path.join(MODEL_DIR, f"{MODEL_NAME}_best.pth")
model.load_state_dict(torch.load(best_path, map_location=device))
print(f"Loaded best model from {best_path}")

plot_parity(val_trajs, model, device, MODEL_NAME, PNG_DIR)

## 12 — Test Set Evaluation
Run once at the end with the best checkpoint. Never use test set to guide hyperparameter choices.

In [ ]:
# Label test set with flat terminal results (ground truth only, no TD blending)
test_ds.relabel_flat()
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

loss_fn    = nn.HuberLoss(delta=0.2)
test_loss  = run_epoch(test_loader, model, None, loss_fn, device, train=False)
test_acc   = directional_accuracy(test_loader, model, device)

print(f"Test Huber loss           : {test_loss:.4f}")
print(f"Test directional accuracy : {100*test_acc:.1f}%")

# Parity on test set
plot_parity(test_trajs, model, device, MODEL_NAME, PNG_DIR)